## TASK 2 GEN AI
ifrah imran

In [ ]:
# 1. Install Unsloth and essential dependencies
!pip install unsloth
!pip install --no-deps "xformers<0.0.29" "trl<0.13.0" peft accelerate bitsandbytes

# 2. Import the specialized Fast Language Model
from unsloth import FastLanguageModel
import torch

# 3. Configuration
max_seq_length = 2048 # Supports RoPE Scaling internally
dtype = None # Auto-detection (Float16 for Tesla T4, Bfloat16 for A100)
load_in_4bit = True # Essential for Colab's 16GB VRAM limit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.8/54.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 MB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 402.9/402.9 kB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 71.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 71.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 75.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.9

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit", # Or "unsloth/DeepSeek-R1-Distill-Llama-8B"
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Configure the LoRA parameters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank: higher = more parameters, lower = more efficient
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Optimized to 0 for Unsloth
    bias = "none",    # Optimized to "none" for Unsloth
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

==((====))==  Unsloth 2026.3.11: Fast Llama patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/198 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.
Unsloth 2026.3.11 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [ ]:
#3
from datasets import load_dataset

# 1. Load the medical reasoning dataset
dataset = load_dataset("FreedomIntelligence/Medical-O1-Reasoning-SFT", "en", split = "train[:500]")

# 2. Format the data into a prompt the model understands
standard_prompt = """### Question:
{}

### Thinking:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN

def formatting_prompts_func(examples):
    inputs       = examples["Question"]
    complex_cot  = examples["Complex_CoT"]
    outputs      = examples["Response"]
    texts = []
    for input, cot, output in zip(inputs, complex_cot, outputs):
        # Must add EOS_TOKEN, otherwise generation will go on forever!
        t = standard_prompt.format(input, cot, output) + EOS_TOKEN
        texts.append(t)
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True)

README.md: 0.00B [00:00, ?B/s]

medical_o1_sft.json:   0%|          | 0.00/58.2M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
import torch
import gc

# 1. Clear Python's garbage collector
gc.collect()

# 2. Clear the CUDA cache
torch.cuda.empty_cache()

# 3. Check if memory is actually free now
print(f"Memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"Memory reserved: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

Memory allocated: 5.97 GB
Memory reserved: 6.10 GB


In [ ]:
# --- PHASE 4: ABSOLUTE MINIMUM VRAM WORKFLOW ---
from trl import SFTTrainer, SFTConfig
from transformers import DataCollatorForLanguageModeling
import torch

# 1. Force clear memory and fragmentation
torch.cuda.empty_cache()

# 2. Setup the data collator
collator = DataCollatorForLanguageModeling(tokenizer = tokenizer, mlm = False)

# 3. Aggressive Memory Configuration
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 256,          # Dropped further to 256 for absolute safety
    data_collator = collator,
    args = SFTConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 16,
        warmup_steps = 2,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = True,
        logging_steps = 1,
        optim = "paged_adamw_8bit", # 'paged' allows offloading to CPU RAM if GPU is full
        weight_decay = 0.01,
        seed = 3407,
        output_dir = "outputs",
        gradient_checkpointing = True,
        # This helps with the 'Fused Loss' OOM you just saw
        gradient_checkpointing_kwargs = {"use_reentrant": False},
    ),
)

# 4. Unsloth's secret weapon for OOM
from unsloth import FastLanguageModel
model = FastLanguageModel.for_training(model)

print("\n🚀 Starting Absolute Minimum VRAM Training...")
trainer_stats = trainer.train()

# 5. Save progress
model.save_pretrained("medical_ai_adapter")
tokenizer.save_pretrained("medical_ai_adapter")

/tmp/unsloth_compiled_cache/UnslothSFTTrainer.py:860: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/tmp/unsloth_compiled_cache/UnslothSFTTrainer.py:888: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(



🚀 Starting Absolute Minimum VRAM Training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 2 | Total steps = 60
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss
1,1.401679
2,1.365810
3,1.431569
4,1.438614
5,1.302569
6,1.349854
7,1.346354
8,1.344215
9,1.333685
10,1.351660


('medical_ai_adapter/tokenizer_config.json',
 'medical_ai_adapter/tokenizer.json')

In [ ]:
# --- PHASE 5 KERNEL OVERRIDE ---
import torch
from unsloth.models.llama import LlamaAttention_fast_forward_inference

# 1. THE RESET: Force the model to forget the "Fast" inference kernels
# This points the failing function to None, forcing a fallback to standard PyTorch
import unsloth.models.llama
unsloth.models.llama.LlamaAttention_fast_forward_inference = None

# 2. Prepare for inference
model.eval()
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

# 3. Define the prompt
prompt = "### Question:\nA 45-year-old male presents with sudden chest pain and shortness of breath. What are the immediate diagnostic steps?\n\n### Thinking:\n"
inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

# 4. Generate using the most basic  path
print("\n🚀 Starting Forced-Stable Generation...")
with torch.no_grad():
    outputs = model.generate(
        input_ids = inputs.input_ids,
        attention_mask = inputs.attention_mask,
        max_new_tokens = 256,
        use_cache = False,   # Disabling cache avoids the KV-shape mismatch entirely
        do_sample = False,    # Greedy search for maximum stability
        position_ids = None
    )

# 5. Decode
result = tokenizer.batch_decode(outputs, skip_special_tokens = True)[0]
print("\n" + "="*50)
print(" MEDICAL AI DIAGNOSIS:")
print("="*50)
print(result)
print("="*50)

Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🚀 Starting Forced-Stable Generation...


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)



🩺 MEDICAL AI DIAGNOSIS:
### Question:
A 45-year-old male presents with sudden chest pain and shortness of breath. What are the immediate diagnostic steps?

### Thinking:
Okay, so we have a 45-year-old guy who's suddenly feeling chest pain and having trouble breathing. That's definitely concerning. Chest pain and shortness of breath are classic signs of a heart attack or something going on with the heart. Let's think about what we need to do first.

First things first, we need to stabilize this person. We can't waste time. We need to make sure he's breathing well and that his heart is stable. If he's in immediate danger, we might need to start CPR or use a defibrillator right away. But let's hope it doesn't come to that.

Now, we need to figure out what's going on. We need to get a quick look at his heart. An ECG is a must. It's like a snapshot of the heart's electrical activity, and it can tell us if there's a heart attack or some other serious issue. We can do this right away.

Oh, a